# Inference time benchmark OPT-1.3B + LoRA adapters

Applied to the 3 LoRA adapters

In [2]:
import time
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

/user/HS401/mf01425/Documents/coursework/Waste-Classification-main/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
short_text  = "Oh brilliant, just what I needed."
medium_text = "The service was absolutely fantastic, I would recommend to everyone."
long_text = "The service was absolutely fantastic and I cannot begin to express how thoroughly impressed I was with every single aspect of my visit to this establishment, from the moment I walked through the door to the time I left, everything was handled with such professionalism and care that I would without hesitation recommend this place to absolutely everyone I know and have ever met in my entire life, it was truly a remarkable and unforgettable experience that exceeded all of my expectations completely."

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {DEVICE}")

Using: cuda


In [5]:
def time_model(predict_fn, text, n_runs=20, n_warmup=3):
    for _ in range(n_warmup):
        predict_fn(text)

    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        predict_fn(text)
        end = time.perf_counter()
        times.append((end - start) * 1000)
    return {
        "mean_ms": round(np.mean(times), 2),
        "std_ms":  round(np.std(times), 2)
    }

## Load OPT-1.3B base + 3 LoRA adapters

The base model is loaded once. The three adapters are attached to the same
base via PEFT's multi-adapter API. Switching variety at inference time uses
`set_adapter()` which is constant-time (no model reload).

In [6]:
BASE_MODEL = "facebook/opt-1.3b"
ADAPTERS = {
    "en-UK": "momofahmi/besstie-lora-en-uk-opt-1.3b",
    "en-AU": "momofahmi/besstie-lora-en-au-opt-1.3b",
    "en-IN": "momofahmi/besstie-lora-en-in-opt-1.3b",
}

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2, dtype=dtype,
)
base_model.config.pad_token_id = tokenizer.pad_token_id

peft_model = PeftModel.from_pretrained(base_model, ADAPTERS["en-UK"], adapter_name="en-UK")
peft_model.load_adapter(ADAPTERS["en-AU"], adapter_name="en-AU")
peft_model.load_adapter(ADAPTERS["en-IN"], adapter_name="en-IN")
peft_model.eval()
peft_model = peft_model.to(DEVICE)
print("All 3 adapters loaded")

Loading weights: 100%|██████████| 388/388 [00:00<00:00, 92083.40it/s]
OPTForSequenceClassification LOAD REPORT from: facebook/opt-1.3b
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


All 3 adapters loaded


## Inference function

In [7]:
def predict_opt(text, variety):
    peft_model.set_adapter(variety)

    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=128
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = peft_model(**inputs)
    return torch.argmax(outputs.logits).item()

## Benchmark 3 texts × 3 adapters

In [8]:
# en-UK adapter
print("─── OPT-1.3B + LoRA en-UK ───")
print("Short :", time_model(lambda t: predict_opt(t, "en-UK"), short_text))
print("Medium:", time_model(lambda t: predict_opt(t, "en-UK"), medium_text))
print("Long  :", time_model(lambda t: predict_opt(t, "en-UK"), long_text))

`use_return_dict` is deprecated! Use `return_dict` instead!


─── OPT-1.3B + LoRA en-UK ───
Short : {'mean_ms': np.float64(12.32), 'std_ms': np.float64(0.52)}
Medium: {'mean_ms': np.float64(12.2), 'std_ms': np.float64(0.38)}
Long  : {'mean_ms': np.float64(15.15), 'std_ms': np.float64(0.32)}


In [9]:
# en-AU adapter
print("─── OPT-1.3B + LoRA en-AU ───")
print("Short :", time_model(lambda t: predict_opt(t, "en-AU"), short_text))
print("Medium:", time_model(lambda t: predict_opt(t, "en-AU"), medium_text))
print("Long  :", time_model(lambda t: predict_opt(t, "en-AU"), long_text))

─── OPT-1.3B + LoRA en-AU ───
Short : {'mean_ms': np.float64(12.01), 'std_ms': np.float64(0.35)}
Medium: {'mean_ms': np.float64(12.19), 'std_ms': np.float64(0.35)}
Long  : {'mean_ms': np.float64(15.13), 'std_ms': np.float64(0.34)}


In [10]:
# en-IN adapter
print("─── OPT-1.3B + LoRA en-IN ───")
print("Short :", time_model(lambda t: predict_opt(t, "en-IN"), short_text))
print("Medium:", time_model(lambda t: predict_opt(t, "en-IN"), medium_text))
print("Long  :", time_model(lambda t: predict_opt(t, "en-IN"), long_text))

─── OPT-1.3B + LoRA en-IN ───
Short : {'mean_ms': np.float64(11.96), 'std_ms': np.float64(0.37)}
Medium: {'mean_ms': np.float64(12.11), 'std_ms': np.float64(0.36)}
Long  : {'mean_ms': np.float64(15.08), 'std_ms': np.float64(0.32)}
